# Tiny transformer, by hand

This notebook prints every self-attention calculation for `river bank` and `bank river`, followed by the output projection and first skip connection. It compares attention with and without positional embeddings so you can see exactly where word order enters the model.

In [1]:
from math import isinf

from tiny_transformer import AttentionTrace, teaching_model

## Display helpers

The model returns every intermediate matrix in an `AttentionTrace`. These helpers round the numbers and make masked scores easy to recognize.

In [2]:
def rounded(matrix: list[list[float]]) -> list[list[float | str]]:
    return [
        ["masked" if isinf(number) else round(number, 3) for number in row]
        for row in matrix
    ]


def show(trace: AttentionTrace) -> None:
    print(f"tokens:  {trace.tokens}")
    print(f"X:       {rounded(trace.inputs)}")
    print(f"Q:       {rounded(trace.queries)}")
    print(f"K:       {rounded(trace.keys)}")
    print(f"V:       {rounded(trace.values)}")
    print(f"QK^T:    {rounded(trace.scores)}")
    print(f"softmax: {rounded(trace.weights)}")
    print(f"attention: {rounded(trace.outputs)}")
    print(f"@ W_O:     {rounded(trace.projected_outputs)}")
    print(f"+ X:       {rounded(trace.residual_outputs)}")

## Output projection and skip connection

`W_O` means the attention output matrix. After attention mixes the values, the mixed vector is multiplied by `W_O`. In this model `W_O` is the identity matrix, so the numbers do not change. The skip connection then adds the original input `X`.

For the first token in causal `river bank`:

```text
attention output    = [1, 0]
projected attention = [1, 0] @ W_O = [1, 0]
original input      = [1, 0]
after skip          = [1, 0] + [1, 0] = [2, 0]
```

## Configure the walkthrough

Set `causal` to `True` to prevent each token from attending to tokens that come after it.

In [4]:
causal = True # False
model = teaching_model()

## Compare word order

Without positions, swapping the words only reorders the result. With positions, each word receives a different input vector—and therefore different queries, keys, and values—depending on where it appears.

In [5]:
for use_positions in (False, True):
    label = "WITH positions" if use_positions else "WITHOUT positions"
    print(f"\n=== {label} ===")
    for tokens in (["river", "bank"], ["bank", "river"]):
        print()
        show(
            model.forward(
                tokens,
                use_positions=use_positions,
                causal=causal,
            )
        )

## Try it yourself

Change the second position vector in `teaching_model()` or swap the two coordinates of an embedding, then restart the kernel and run all cells to see which calculations change.